In [82]:
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [83]:
def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False


FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")

try:
    import faiss  # type: ignore
except Exception:
    faiss = None
    FAISS_READY = False


# sentence-transformers опционален: ноутбук умеет работать и без него.
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS доступен:", FAISS_READY)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)

NumPy: 2.2.6
Pandas: 3.0.2
FAISS доступен: True
sentence-transformers доступен: True


In [84]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

try:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


In [85]:
df = pd.read_json("hf://datasets/MakTek/Customer_support_faqs_dataset/train_expanded.json", lines=True)
df

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...
...,...,...
195,Do you offer a satisfaction guarantee?,"Yes, we offer a satisfaction guarantee on our ..."
196,How can I apply for a job at your company?,"To apply for a job at our company, visit our C..."
197,What is the warranty on your products?,The warranty on our products varies by item. P...
198,Can I request a refund if the price drops afte...,If the price of a product drops within 7 days ...


In [86]:
df_docs = df[['answer']].head(20).copy()
df_docs.insert(0, 'id', range(len(df_docs)))
display(df_docs)

,id,answer
0,0,"To create an account, click on the 'Sign Up' b..."
1,1,"We accept major credit cards, debit cards, and..."
2,2,You can track your order by logging into your ...
3,3,Our return policy allows you to return product...
4,4,You can cancel your order if it has not been s...
5,5,Shipping times vary depending on the destinati...
6,6,"Yes, we offer international shipping to select..."
7,7,If your package is lost or damaged during tran...
8,8,"If you need to change your shipping address, p..."
9,9,You can contact our customer support team by p...


In [87]:
print(f"\nЧисло документов: {len(df_docs)}")
for idx in range(3):
    print(f"Документ {idx}:")
    print(df_docs.iloc[idx]['answer'])


Число документов: 20
Документ 0:
To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.
Документ 1:
We accept major credit cards, debit cards, and PayPal as payment methods for online orders.
Документ 2:
You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.


База знаний - FAQ поддержки интернет магазина. Хороший вариант для retrieval / mini-RAG, потому что ее можно использовать для обучения чат-бота поддержки.

In [88]:
def chunk_text(text: str, chunk_size: int = 28, overlap: int = 8) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks

class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF (fallback)"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.fit_transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    # Опциональная попытка dense backend.
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend(
                model_name="all-MiniLM-L6-v2",
                device=device,
            )
        except Exception as e:
            print("Dense backend недоступен, переходим к TF-IDF.")
            print("Причина:", repr(e))
    return TfidfBackend()

@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object

def build_retriever(
    documents: pd.DataFrame,
    chunk_size: int = 28,
    overlap: int = 8,
    device: str = "cpu",
) -> RetrieverArtifacts:
    rows = []
    for _, doc in documents.iterrows():
        chunks = chunk_text(doc["answer"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk_text_value in enumerate(chunks, start=1):
            rows.append(
                {
                    "doc_id": doc["id"],
                    "chunk_id": f'{doc["id"]}_chunk_{chunk_id:02d}',
                    "chunk_text": chunk_text_value,
                }
            )

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist()).astype(np.float32)

    if FAISS_READY:
        index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # type: ignore
        index.add(chunk_vectors)
    else:
        index = chunk_vectors

    return RetrieverArtifacts(
        backend_name=backend.backend_name,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        backend=backend,
        index=index,
    )

def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype(np.float32)

    if FAISS_READY:
        scores, indices = artifacts.index.search(query_vector, top_k)  # type: ignore
        scores = scores[0]
        indices = indices[0]
    else:
        similarities = (artifacts.chunk_vectors @ query_vector.T).reshape(-1)
        indices = np.argsort(-similarities)[:top_k]
        scores = similarities[indices]

    result = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores
    return result[["rank", "score", "doc_id", "chunk_id", "chunk_text"]]

In [89]:
artifacts = build_retriever(
    df_docs,
    chunk_size=16,
    overlap=4,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5559.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используемый backend: SentenceTransformer: all-MiniLM-L6-v2
Количество чанков: 50


,doc_id,chunk_id,chunk_text
0,0,0_chunk_01,"To create an account, click on the 'Sign Up' b..."
1,0,0_chunk_02,top right corner of our website and follow the...
2,1,1_chunk_01,"We accept major credit cards, debit cards, and..."
3,2,2_chunk_01,You can track your order by logging into your ...
4,2,2_chunk_02,"to the 'Order History' section. There, you wil..."


In [90]:
embedder = choose_backend(device=DEVICE)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4628.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [91]:
chunk_texts = artifacts.chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))

Форма матрицы эмбеддингов: (50, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0


In [92]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_READY:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices

search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


In [93]:
def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "chunk_id": chunk_row["chunk_id"],
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)

faiss_query = ["How can I track my order?",
    "Can I cancel my order?",
    "Do you offer gift wrapping services?"]

for query in faiss_query:
    faiss_results_df = search_similar_chunks(query, top_k=5)
    display(Markdown(f"**Запрос:** {query}"))
    display(faiss_results_df)

**Запрос:** How can I track my order?

,rank,doc_id,chunk_id,score,chunk_text
0,1,2,2_chunk_01,0.8514,You can track your order by logging into your ...
1,2,2,2_chunk_02,0.6999,"to the 'Order History' section. There, you wil..."
2,3,12,12_chunk_02,0.5162,order through our website for a smooth and sec...
3,4,4,4_chunk_02,0.5119,Please contact our customer support team with ...
4,5,14,14_chunk_03,0.4952,support team with your order details to reques...


**Запрос:** Can I cancel my order?

,rank,doc_id,chunk_id,score,chunk_text
0,1,4,4_chunk_01,0.7991,You can cancel your order if it has not been s...
1,2,18,18_chunk_01,0.7040,If you need to change or cancel an item in you...
2,3,12,12_chunk_01,0.6051,"Unfortunately, we do not accept orders over th..."
3,4,4,4_chunk_03,0.5904,will assist you with the cancellation process.
4,5,4,4_chunk_02,0.4767,Please contact our customer support team with ...


**Запрос:** Do you offer gift wrapping services?

,rank,doc_id,chunk_id,score,chunk_text
0,1,10,10_chunk_01,0.8043,"Yes, we offer gift wrapping services for an ad..."
1,2,10,10_chunk_02,0.6967,"checkout process, you can select the option to..."
2,3,17,17_chunk_01,0.4511,"Yes, we offer bulk or wholesale discounts for ..."
3,4,12,12_chunk_01,0.3480,"Unfortunately, we do not accept orders over th..."
4,5,3,3_chunk_03,0.3433,and packaging. Please refer to our Returns pag...


In [94]:
def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(str(doc_id))
    return ordered

def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrieverArtifacts,
    top_k: int = 5,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }

def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "relevant_doc_ids": ", ".join(row["relevant_doc_ids"]),
                "predicted_doc_ids": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "first_relevant_rank": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [95]:
benchmark_queries = [
    {"query_id": 0, "query": "Can I pay with a credit card?", "relevant_doc_ids": ["1"]},
    {"query_id": 1, "query": "Do you have an expedited delivery?", "relevant_doc_ids": ["5"]},
    {"query_id": 2, "query": "How much does international shipping cost?", "relevant_doc_ids": ["6"]},
    {"query_id": 3, "query": "Can I place an order by phone?", "relevant_doc_ids": ["12"]},
    {"query_id": 4, "query": "Where can I see product reviews?", "relevant_doc_ids": ["19"]},
    {"query_id": 5, "query": "Where is the register button?", "relevant_doc_ids": ["0"]},
    {"query_id": 6, "query": "who should I call to change the delivery address?", "relevant_doc_ids": ["9"]},
    {"query_id": 7, "query": "Do you have promo codes or a loyalty program?", "relevant_doc_ids": ["15"]},
    {"query_id": 8, "query": "What should I do if I received the wrong product?", "relevant_doc_ids": ["18"]}
]

benchmark_df = pd.DataFrame(benchmark_queries)

In [96]:
baseline_eval_1 = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=5)
display(baseline_eval_1)

summary_1 = pd.DataFrame(
    {
        "metric": ["mean_hit@5", "mean_recall@5", "mean_MRR@5"],
        "value": [
            baseline_eval_1["hit@5"].mean(),
            baseline_eval_1["recall@5"].mean(),
            baseline_eval_1["MRR@5"].mean(),
        ],
    }
)
display(summary_1)

,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@5,recall@5,MRR@5,first_relevant_rank
0,0,Can I pay with a credit card?,1,"1, 13, 15, 16, 10",1,1.0,1.0,1.0
1,1,Do you have an expedited delivery?,5,"8, 12, 4",0,0.0,0.0,NaN
2,2,How much does international shipping cost?,6,"6, 5, 8",1,1.0,1.0,1.0
3,3,Can I place an order by phone?,12,"12, 16, 2, 4, 8",1,1.0,1.0,1.0
4,4,Where can I see product reviews?,19,"19, 11, 17",1,1.0,1.0,1.0
5,5,Where is the register button?,0,"0, 19, 16",1,1.0,1.0,1.0
6,6,who should I call to change the delivery address?,9,"8, 18, 14",0,0.0,0.0,NaN
7,7,Do you have promo codes or a loyalty program?,15,"15, 17, 14, 11",1,1.0,1.0,1.0
8,8,What should I do if I received the wrong product?,18,"9, 7, 4, 11",0,0.0,0.0,NaN


,metric,value
0,mean_hit@5,0.666667
1,mean_recall@5,0.666667
2,mean_MRR@5,0.666667


In [97]:
artifacts_2 = build_retriever(
    df_docs,
    chunk_size=8,
    overlap=4,
    device=DEVICE,
)

baseline_eval_2 = evaluate_benchmark(benchmark_queries, artifacts=artifacts_2, top_k=5)
display(baseline_eval_2)

summary_2 = pd.DataFrame(
    {
        "metric": ["mean_hit@5", "mean_recall@5", "mean_MRR@5"],
        "value": [
            baseline_eval_2["hit@5"].mean(),
            baseline_eval_2["recall@5"].mean(),
            baseline_eval_2["MRR@5"].mean(),
        ],
    }
)
display(summary_2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2681.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@5,recall@5,MRR@5,first_relevant_rank
0,0,Can I pay with a credit card?,1,"1, 10, 13",1,1.0,1.00,1.0
1,1,Do you have an expedited delivery?,5,"8, 4, 12, 5",1,1.0,0.25,4.0
2,2,How much does international shipping cost?,6,"6, 5",1,1.0,1.00,1.0
3,3,Can I place an order by phone?,12,"12, 16, 2, 4",1,1.0,1.00,1.0
4,4,Where can I see product reviews?,19,19,1,1.0,1.00,1.0
5,5,Where is the register button?,0,"0, 19",1,1.0,1.00,1.0
6,6,who should I call to change the delivery address?,9,"8, 12",0,0.0,0.00,NaN
7,7,Do you have promo codes or a loyalty program?,15,"15, 17",1,1.0,1.00,1.0
8,8,What should I do if I received the wrong product?,18,"17, 9, 3, 19, 18",1,1.0,0.20,5.0


,metric,value
0,mean_hit@5,0.888889
1,mean_recall@5,0.888889
2,mean_MRR@5,0.716667


Я провела эксперимент с двумя разными chunk_size - 16 и 8. По результатам видно, что меньший chunk_size дал лучшие метрики. Думаю, это связано с тем, что документы в моей базе сами по себе небольшие.

In [98]:
retrieval_eval_data = []

for idx, row in baseline_eval_2.iterrows():
    retrieval_eval_data.append({
        'query': row['query'],
        'expected_source': row["relevant_doc_ids"],
        'retrieved_sources': row["predicted_doc_ids"],
        'hit_at_k': int(row['hit@5'])
    })

retrieval_eval_df = pd.DataFrame(retrieval_eval_data)
retrieval_eval_df.to_csv('artifacts/retrieval_eval.csv', index=False)
display(retrieval_eval_df)

,query,expected_source,retrieved_sources,hit_at_k
0,Can I pay with a credit card?,1,"1, 10, 13",1
1,Do you have an expedited delivery?,5,"8, 4, 12, 5",1
2,How much does international shipping cost?,6,"6, 5",1
3,Can I place an order by phone?,12,"12, 16, 2, 4",1
4,Where can I see product reviews?,19,19,1
5,Where is the register button?,0,"0, 19",1
6,who should I call to change the delivery address?,9,"8, 12",0
7,Do you have promo codes or a loyalty program?,15,"15, 17",1
8,What should I do if I received the wrong product?,18,"17, 9, 3, 19, 18",1


In [99]:
df = pd.read_json("hf://datasets/MakTek/Customer_support_faqs_dataset/train_expanded.json", lines=True)

df_docs_new = df[['answer']].iloc[20:25].copy()
df_docs_new.insert(0, 'id', range(len(df_docs), len(df_docs) + len(df_docs_new)))

# Объединяем с предыдущим df_docs
df_docs_updated = pd.concat([df_docs, df_docs_new], ignore_index=False).reset_index(drop=True)

print(f"Новое количество документов: {len(df_docs_updated)}")
display(df_docs_updated)

Новое количество документов: 25


,id,answer
0,0,"To create an account, click on the 'Sign Up' b..."
1,1,"We accept major credit cards, debit cards, and..."
2,2,You can track your order by logging into your ...
3,3,Our return policy allows you to return product...
4,4,You can cancel your order if it has not been s...
5,5,Shipping times vary depending on the destinati...
6,6,"Yes, we offer international shipping to select..."
7,7,If your package is lost or damaged during tran...
8,8,"If you need to change your shipping address, p..."
9,9,You can contact our customer support team by p...


In [100]:
queries = [{"query_id": 1, "query": "Do you have an expedited delivery?", "relevant_doc_ids": ["22"]},
    {"query_id": 7, "query": "Do you have promo codes or a loyalty program?", "relevant_doc_ids": ["20"]},
    {"query_id": 8, "query": "What should I do if I received the wrong product?", "relevant_doc_ids": ["21"]}
]

for query in queries:
    display(Markdown(f"**Запрос:** {query['query']}"))
    display(search_chunks(query['query'], artifacts=artifacts, top_k=5)[["rank", "score", "doc_id", "chunk_id", "chunk_text"]])

**Запрос:** Do you have an expedited delivery?

,rank,score,doc_id,chunk_id,chunk_text
0,1,0.470851,8,8_chunk_03,update the address if the order has not been s...
1,2,0.443621,12,12_chunk_01,"Unfortunately, we do not accept orders over th..."
2,3,0.438770,4,4_chunk_01,You can cancel your order if it has not been s...
3,4,0.411697,8,8_chunk_01,"If you need to change your shipping address, p..."
4,5,0.408526,12,12_chunk_02,order through our website for a smooth and sec...


**Запрос:** Do you have promo codes or a loyalty program?

,rank,score,doc_id,chunk_id,chunk_text
0,1,0.579514,15,15_chunk_01,"Yes, we have a loyalty program where you can e..."
1,2,0.515925,15,15_chunk_02,every purchase. These points can be redeemed f...
2,3,0.444335,17,17_chunk_01,"Yes, we offer bulk or wholesale discounts for ..."
3,4,0.388933,14,14_chunk_01,If a product you purchased goes on sale within...
4,5,0.374662,11,11_chunk_03,customer support team with the details of the ...


**Запрос:** What should I do if I received the wrong product?

,rank,score,doc_id,chunk_id,chunk_text
0,1,0.477743,9,9_chunk_01,You can contact our customer support team by p...
1,2,0.458131,7,7_chunk_02,customer support team immediately. We will ini...
2,3,0.447028,4,4_chunk_01,You can cancel your order if it has not been s...
3,4,0.433479,11,11_chunk_02,of an identical product found on a competitor'...
4,5,0.420746,4,4_chunk_02,Please contact our customer support team with ...


In [101]:
updated_queries = [
    {"query_id": 0, "query": "Can I pay with a credit card?", "relevant_doc_ids": ["1"]},
    {"query_id": 1, "query": "Do you have an expedited delivery?", "relevant_doc_ids": ["22"]},
    {"query_id": 2, "query": "How much does international shipping cost?", "relevant_doc_ids": ["6"]},
    {"query_id": 3, "query": "Can I place an order by phone?", "relevant_doc_ids": ["12"]},
    {"query_id": 4, "query": "Where can I see product reviews?", "relevant_doc_ids": ["19"]},
    {"query_id": 5, "query": "Where is the register button?", "relevant_doc_ids": ["0"]},
    {"query_id": 6, "query": "who should I call to change the delivery address?", "relevant_doc_ids": ["9"]},
    {"query_id": 7, "query": "Do you have promo codes or a loyalty program?", "relevant_doc_ids": ["20"]},
    {"query_id": 8, "query": "What should I do if I received the wrong product?", "relevant_doc_ids": ["21"]}
]

In [102]:
artifacts_3 = build_retriever(
    df_docs_updated,
    chunk_size=16,
    overlap=4,
    device=DEVICE,
)

baseline_eval_3 = evaluate_benchmark(updated_queries, artifacts=artifacts_3, top_k=5)
display(baseline_eval_3)

summary_3 = pd.DataFrame(
    {
        "metric": ["mean_hit@5", "mean_recall@5", "mean_MRR@5"],
        "value": [
            baseline_eval_3["hit@5"].mean(),
            baseline_eval_3["recall@5"].mean(),
            baseline_eval_3["MRR@5"].mean(),
        ],
    }
)
display(summary_3)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2999.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@5,recall@5,MRR@5,first_relevant_rank
0,0,Can I pay with a credit card?,1,"1, 13, 15, 16, 10",1,1.0,1.0,1.0
1,1,Do you have an expedited delivery?,22,"22, 8, 12, 4",1,1.0,1.0,1.0
2,2,How much does international shipping cost?,6,"6, 5, 8",1,1.0,1.0,1.0
3,3,Can I place an order by phone?,12,"12, 16, 2, 4, 8",1,1.0,1.0,1.0
4,4,Where can I see product reviews?,19,"19, 11, 24",1,1.0,1.0,1.0
5,5,Where is the register button?,0,"0, 20, 23, 19",1,1.0,1.0,1.0
6,6,who should I call to change the delivery address?,9,"8, 18, 21",0,0.0,0.0,NaN
7,7,Do you have promo codes or a loyalty program?,20,"20, 15, 17",1,1.0,1.0,1.0
8,8,What should I do if I received the wrong product?,21,"21, 23, 9",1,1.0,1.0,1.0


,metric,value
0,mean_hit@5,0.888889
1,mean_recall@5,0.888889
2,mean_MRR@5,0.888889


In [103]:
comparison_queries = [
    "Do you have an expedited delivery?",
    "Do you have promo codes or a loyalty program?",
    "What should I do if I received the wrong product?"
]

comparison_rows = []
for query in comparison_queries:
    before_df = search_chunks(query, artifacts=artifacts, top_k=5)
    after_df = search_chunks(query, artifacts=artifacts_3, top_k=5)

    before_sources = "; ".join(before_df["doc_id"].astype(str).tolist())
    after_sources = "; ".join(after_df["doc_id"].astype(str).tolist())

    comparison_rows.append({
        "query": query,
        "before_retrieved_sources": before_sources,
        "after_retrieved_sources": after_sources,
        "changed": before_sources != after_sources,
    })

retrieval_before_after_update_df = pd.DataFrame(comparison_rows)
retrieval_before_after_update_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
display(retrieval_before_after_update_df)

,query,before_retrieved_sources,after_retrieved_sources,changed
0,Do you have an expedited delivery?,8; 12; 4; 8; 12,22; 22; 8; 12; 4,True
1,Do you have promo codes or a loyalty program?,15; 15; 17; 14; 11,20; 20; 15; 15; 17,True
2,What should I do if I received the wrong product?,9; 7; 4; 11; 4,21; 21; 23; 21; 9,True


Я обновила базу знаний, повторно выполнила чанкинг и задала 3 вопроса, более точные ответы на которые присутствуют в новых документах. Как видно, теперь модель более точно находит ответ.

In [104]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def pick_best_sentences(query: str, text: str, top_n: int = 2) -> List[str]:
    sentences = split_into_sentences(text)
    if not sentences:
        return []

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentences).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    best_idx = np.argsort(-scores)[:top_n]
    return [sentences[i] for i in best_idx if scores[i] > 0]

In [105]:
def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 5) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [106]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [107]:
def mini_rag_answer(
    query: str,
    artifacts: RetrieverArtifacts,
    top_k: int = 5,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }


In [108]:
rag_quiries = [
    "Do you have an expedited delivery?",
    "Where is the register button?",
    "Do you have promo codes or a loyalty program?"
]

rag_examples_data = []

for query in rag_quiries:
    rag_result = mini_rag_answer(
        query,
        artifacts=artifacts,
        top_k=5,
    )

    sources = rag_result.get('sources', [])
    if isinstance(sources, pd.DataFrame):
        sources_str = "; ".join(sources["doc_id"].astype(str).tolist()) if not sources.empty else "N/A"
    else:
        sources_str = "; ".join([str(doc_id) for doc_id in sources]) if sources else "N/A"
    
    rag_examples_data.append({
        'question': query,
        'answer': rag_result.get('answer', ''),
        'retrieved_sources': sources_str
    })

    display(Markdown(f"### Вопрос: {rag_result['query']}"))
    display(Markdown(f"**Ответ:** {rag_result['answer']}"))
    display(Markdown("**Источники:**"))
    display(rag_result["sources"])

rag_examples_df = pd.DataFrame(rag_examples_data)

### Вопрос: Do you have an expedited delivery?

**Ответ:** Unfortunately, we do not accept orders over the phone. You can cancel your order if it has not been shipped yet.

**Источники:**

,rank,score,doc_id,chunk_id,chunk_text
0,1,0.470851,8,8_chunk_03,update the address if the order has not been s...
1,2,0.443621,12,12_chunk_01,"Unfortunately, we do not accept orders over th..."
2,3,0.438770,4,4_chunk_01,You can cancel your order if it has not been s...
3,4,0.411697,8,8_chunk_01,"If you need to change your shipping address, p..."
4,5,0.408526,12,12_chunk_02,order through our website for a smooth and sec...


### Вопрос: Where is the register button?

...

,rank,score,doc_id,chunk_id,chunk_text
0,1,0.656917,0,0_chunk_02,top right corner of our website and follow the...
1,2,0.483528,0,0_chunk_01,"To create an account, click on the 'Sign Up' b..."
2,3,0.260568,19,19_chunk_01,"To leave a product review, navigate to the pro..."
3,4,0.259570,19,19_chunk_02,website and click on the 'Write a Review' butt...
4,5,0.233929,16,16_chunk_02,"account. However, creating an account offers b..."


### Вопрос: Do you have promo codes or a loyalty program?

**Ответ:** Yes, we have a loyalty program where you can earn points for every purchase. Yes, we offer bulk or wholesale discounts for certain products.

**Источники:**

,rank,score,doc_id,chunk_id,chunk_text
0,1,0.579514,15,15_chunk_01,"Yes, we have a loyalty program where you can e..."
1,2,0.515925,15,15_chunk_02,every purchase. These points can be redeemed f...
2,3,0.444335,17,17_chunk_01,"Yes, we offer bulk or wholesale discounts for ..."
3,4,0.388933,14,14_chunk_01,If a product you purchased goes on sale within...
4,5,0.374662,11,11_chunk_03,customer support team with the details of the ...


Ответ на 2 вопрос оказался неудачным, в вопросе не было ничего про промокоды. Ошибка произошла из-за того, что есть документ про промокоды, в котором также упоминается специальное поле.

In [109]:
rag_examples_df.to_csv('artifacts/rag_examples.csv', index=False)